# 01 · Audio basics

How sound becomes the numbers EchoTTS learns from. Run each cell top to bottom. If LJSpeech is in `data/LJSpeech-1.1/` it uses a real clip; otherwise it makes a synthetic 'vowel' so the notebook still runs.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio
from echotts.config import load_config
from echotts.audio import load_wav, mel_spectrogram, griffin_lim, _stft_magnitude
cfg = load_config(ROOT / 'configs' / 'base.yaml')['audio']
SR = cfg['sample_rate']

In [ ]:
clip = ROOT / 'data' / 'LJSpeech-1.1' / 'wavs' / 'LJ001-0001.wav'
if clip.exists():
    wav = load_wav(clip, cfg)
else:
    # Fake vowel: a 140 Hz buzz with harmonics, pitch gliding, loudness rising and falling.
    t = torch.arange(int(2.0 * SR)) / SR
    f0 = 120 + 40 * t
    phase = 2 * torch.pi * torch.cumsum(f0, 0) / SR
    wav = sum(torch.sin(k * phase) / k for k in range(1, 15)) * torch.sin(torch.pi * t / 2.0)
    wav = 0.9 * wav / wav.abs().max()
print('sample rate :', SR, 'Hz')
print('samples     :', wav.numel())
print('duration    :', round(wav.numel() / SR, 2), 's')
Audio(wav.numpy(), rate=SR)

## 1. The waveform

A microphone measures air pressure **22,050 times per second** (the *sample rate*). The plot is that list of numbers. **Analogy:** a flip-book where each page is one pressure reading. It shows *loudness over time* but hides *which notes* are playing.

In [ ]:
plt.figure(figsize=(12, 3)); plt.plot(np.arange(wav.numel()) / SR, wav.numpy(), lw=0.4)
plt.xlabel('seconds'); plt.ylabel('amplitude'); plt.title('Waveform'); plt.show()

## 2. The STFT spectrogram

The **Short-Time Fourier Transform** cuts the audio into windows of 1024 samples (~46 ms), sliding forward **256 samples** each time (the *hop length*, ~86 frames per second). For each window it measures how strong each frequency is. **Analogy:** a piano roll — time left to right, pitch bottom to top, brightness = how loud that pitch is.

In [ ]:
mag = _stft_magnitude(wav, cfg)  # (513 freqs, frames)
print('STFT shape:', tuple(mag.shape))
plt.figure(figsize=(12, 4)); plt.imshow(torch.log(mag + 1e-5).numpy(), origin='lower', aspect='auto',
    extent=[0, wav.numel() / SR, 0, SR / 2])
plt.xlabel('seconds'); plt.ylabel('Hz'); plt.title('Log STFT magnitude'); plt.colorbar(); plt.show()

## 3. The mel spectrogram

Ears hear 100→200 Hz as a big jump but 7,000→7,100 Hz as nearly nothing. The **mel scale** groups frequencies the way ears do: 80 triangular filters, narrow at low pitch and wide at high pitch, covering 0–8,000 Hz. Then we take the natural log (clamped at 1e-5) because loudness is also heard on a log scale. **Analogy:** shrinking a 513-key piano down to 80 keys, keeping more keys where your ear is picky. This 80 × frames picture is exactly what the acoustic model will learn to draw.

In [ ]:
mel = mel_spectrogram(wav, cfg)  # (80, frames)
print('mel shape:', tuple(mel.shape), ' samples/256 =', wav.numel() // 256)
plt.figure(figsize=(12, 4)); plt.imshow(mel.numpy(), origin='lower', aspect='auto')
plt.xlabel('frame'); plt.ylabel('mel band'); plt.title('Log-mel spectrogram'); plt.colorbar(); plt.show()

## 4. Back to sound with Griffin-Lim

The mel threw away **phase** (where each wave is in its cycle). Griffin-Lim guesses it: start random, go to audio and back, keep the new phase, force the known loudness, repeat. **Analogy:** redrawing a picture from a description and correcting it again and again. It sounds a bit metallic; the HiFi-GAN vocoder (stage 6) will do better.

In [ ]:
rebuilt = griffin_lim(mel, cfg, n_iter=60)
Audio(rebuilt.numpy(), rate=SR)